# 🚀 Self-Healing AI — LSTM Training (GPU)
Run all cells top-to-bottom. At the end, 3 files will be downloaded automatically:
- `lstm_model.pt`
- `scaler.pkl`
- `threshold.json`

Drop all 3 into your local `SelfHealingAI/models/` folder, then `docker-compose up -d`.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────
!pip install -q torch scikit-learn joblib numpy

In [ ]:
# ── Cell 2: Paste the source modules inline ──────────────────
import os
os.makedirs('models', exist_ok=True)

# --- synthetic_data_generator.py ---
synthetic_data_code = '''
import numpy as np

RNG = np.random.default_rng(42)

def _normal(n):
    rr = RNG.uniform(0.3, 3.0, n)
    er = RNG.uniform(0.00, 0.03, n)
    lat = RNG.uniform(0.05, 0.4, n)
    mem = RNG.uniform(60, 200, n)
    conn = RNG.uniform(2, 30, n)
    return np.stack([rr, er, lat, mem, conn], axis=1)

def _crash500(n):
    rr  = RNG.uniform(0.5, 4.0, n)
    er  = RNG.uniform(0.5, 0.97, n)
    lat = RNG.uniform(0.5, 3.0, n)
    mem = RNG.uniform(80, 300, n)
    conn = RNG.uniform(5, 50, n)
    return np.stack([rr, er, lat, mem, conn], axis=1)

def _deadlock503(n):
    rr  = RNG.uniform(0.1, 1.0, n)
    er  = RNG.uniform(0.6, 0.99, n)
    lat = RNG.uniform(2.0, 8.0, n)
    mem = RNG.uniform(200, 600, n)
    conn = RNG.uniform(20, 100, n)
    return np.stack([rr, er, lat, mem, conn], axis=1)

def _overload503(n):
    rr  = RNG.uniform(5.0, 9.0, n)
    er  = RNG.uniform(0.3, 0.7, n)
    lat = RNG.uniform(1.0, 5.0, n)
    mem = RNG.uniform(300, 800, n)
    conn = RNG.uniform(50, 150, n)
    return np.stack([rr, er, lat, mem, conn], axis=1)

def _timeout504(n):
    rr  = RNG.uniform(0.1, 2.0, n)
    er  = RNG.uniform(0.4, 0.9, n)
    lat = RNG.uniform(4.0, 8.0, n)
    mem = RNG.uniform(100, 400, n)
    conn = RNG.uniform(10, 60, n)
    return np.stack([rr, er, lat, mem, conn], axis=1)

def _bot429(n):
    rr  = RNG.uniform(6.0, 9.0, n)
    er  = RNG.uniform(0.1, 0.5, n)
    lat = RNG.uniform(0.05, 0.3, n)
    mem = RNG.uniform(60, 150, n)
    conn = RNG.uniform(80, 150, n)
    return np.stack([rr, er, lat, mem, conn], axis=1)

def build_dataset(repeat_normal=10, smooth_window=3, verbose=True):
    segments = []
    fault_sizes = [200, 150, 180, 160, 140]
    normal_sizes = [400, 350, 500, 450, 400, 500, 450, 400, 500, 400]

    for i, n in enumerate(normal_sizes):
        segments.append(_normal(n))

    for n in fault_sizes:
        segments.append(_crash500(n))
        segments.append(_normal(n * 2))
        segments.append(_deadlock503(n))
        segments.append(_normal(n * 2))
        segments.append(_overload503(n))
        segments.append(_normal(n * 2))
        segments.append(_timeout504(n))
        segments.append(_normal(n * 2))
        segments.append(_bot429(n))
        segments.append(_normal(n * 2))

    data = np.concatenate(segments, axis=0)

    if smooth_window > 1:
        kernel = np.ones(smooth_window) / smooth_window
        data = np.stack([np.convolve(data[:, i], kernel, mode="same")
                         for i in range(data.shape[1])], axis=1)

    if verbose:
        print(f"[DATAGEN] Total: {data.shape[0]:,} timesteps x {data.shape[1]} features")
        cols = ["request_rate", "error_rate", "p95_latency", "memory_mb", "active_conns"]
        for i, c in enumerate(cols):
            print(f"  {c:20s} min={data[:,i].min():.3f}  max={data[:,i].max():.3f}  mean={data[:,i].mean():.3f}")

    return data
'''

with open('synthetic_data_generator.py', 'w') as f:
    f.write(synthetic_data_code)

print('Modules written.')

In [ ]:
# ── Cell 3: Train the model ──────────────────────────────────
import json, joblib, torch, torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
from synthetic_data_generator import build_dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# ── Model ────────────────────────────────────────────────────
class LSTMForecastModel(nn.Module):
    def __init__(self, input_size=5, hidden_size=128, num_layers=2,
                 dropout=0.25, num_attention_heads=4, attention_dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.attention = nn.MultiheadAttention(hidden_size, num_attention_heads,
                                               dropout=attention_dropout, batch_first=True)
        self.attn_norm  = nn.LayerNorm(hidden_size)
        self.post_norm  = nn.LayerNorm(hidden_size)
        self.dropout    = nn.Dropout(0.2)
        self.fc1        = nn.Linear(hidden_size, 64)
        self.act        = nn.GELU()
        self.fc2        = nn.Linear(64, input_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        attn_out, _ = self.attention(out, out, out)
        out = self.attn_norm(out + attn_out)
        last = out[:, -1, :]
        last = self.post_norm(last)
        last = self.dropout(last)
        return self.fc2(self.act(self.fc1(last)))

# ── Data ─────────────────────────────────────────────────────
SEQ_LEN   = 30
EPOCHS    = 120
BATCH     = 64
PATIENCE  = 15

data   = build_dataset(repeat_normal=10, smooth_window=3, verbose=True)
scaler = StandardScaler()
scaled = scaler.fit_transform(data)

# Build sequences
X_list, y_list = [], []
for i in range(len(scaled) - SEQ_LEN):
    X_list.append(scaled[i:i+SEQ_LEN])
    y_list.append(scaled[i+SEQ_LEN])
X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)
print(f'Sequences: X={X.shape}, y={y.shape}')

X_t, y_t = torch.tensor(X), torch.tensor(y)
ds = TensorDataset(X_t, y_t)
val_n = int(len(ds)*0.2)
tr_ds, va_ds = random_split(ds, [len(ds)-val_n, val_n],
                             generator=torch.Generator().manual_seed(42))
tr_ld = DataLoader(tr_ds, batch_size=BATCH, shuffle=True)
va_ld = DataLoader(va_ds, batch_size=BATCH)

# ── Train ────────────────────────────────────────────────────
model = LSTMForecastModel(input_size=5).to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, patience=7, factor=0.5, min_lr=1e-6)
crit  = nn.MSELoss()

best_val, best_state, patience_cnt, best_epoch = 1e9, None, 0, 0

for epoch in range(1, EPOCHS+1):
    model.train()
    tr_loss = 0
    for bx, by in tr_ld:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optim.zero_grad()
        loss = crit(model(bx), by)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        tr_loss += loss.item()
    tr_loss /= len(tr_ld)

    model.eval()
    va_loss = 0
    with torch.no_grad():
        for bx, by in va_ld:
            va_loss += crit(model(bx.to(DEVICE)), by.to(DEVICE)).item()
    va_loss /= len(va_ld)
    sched.step(va_loss)

    mark = '✓' if va_loss < best_val else ' '
    print(f'[{mark}] Epoch {epoch:3d}/{EPOCHS} | Train: {tr_loss:.6f} | Val: {va_loss:.6f} | Patience: {patience_cnt}/{PATIENCE}')

    if va_loss < best_val:
        best_val, best_state, patience_cnt, best_epoch = va_loss, {k:v.clone() for k,v in model.state_dict().items()}, 0, epoch
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}. Best: {best_epoch}')
            break

print(f'\nBest val loss: {best_val:.6f} at epoch {best_epoch}')

In [ ]:
# ── Cell 4: Compute threshold & save all 3 artifacts ─────────
model.load_state_dict(best_state)
model.eval()

all_mse = []
with torch.no_grad():
    for bx, by in va_ld:
        pred = model(bx.to(DEVICE))
        mse  = torch.mean((pred - by.to(DEVICE))**2, dim=1)
        all_mse.extend(mse.cpu().numpy().tolist())

all_mse   = np.array(all_mse)
threshold = float(np.percentile(all_mse, 99))
print(f'Threshold (p99): {threshold:.6f}')

# Save
torch.save(best_state, 'models/lstm_model.pt')
joblib.dump(scaler,    'models/scaler.pkl')
with open('models/threshold.json', 'w') as f:
    json.dump({'threshold': threshold,
               'method': 'val_mse_p99',
               'best_epoch': best_epoch,
               'best_val_loss': best_val}, f, indent=2)

print('Saved: models/lstm_model.pt, models/scaler.pkl, models/threshold.json')

In [ ]:
# ── Cell 5: Download the 3 files to your PC ──────────────────
from google.colab import files
files.download('models/lstm_model.pt')
files.download('models/scaler.pkl')
files.download('models/threshold.json')
print('Download triggered for all 3 files!')
print()
print('Next steps:')
print('  1. Copy all 3 files into SelfHealingAI/models/')
print('  2. docker-compose up -d')
print('  3. AI inference will start working immediately!')